# Studio Proyek 3 — Fine-Tuning Transformer Pretrained untuk Klasifikasi Sentimen Bahasa Indonesia

| Ketentuan | Detail |
|:---|:---|
| Mata kuliah | Pembelajaran Mendalam (IF25-40401) |
| Institusi | Program Studi Teknik Informatika, Institut Teknologi Sumatera |
| Dataset | SmSA (IndoNLU; Wilie dkk., 2020) — sentimen 3 kelas: *negative*, *neutral*, *positive* |
| Platform | Google Colab / Kaggle (GPU T4 gratis) |
| Batas waktu | 20 November 2026 |

### Gambaran Singkat
Notebook ini membandingkan **tiga backbone** (IndoBERT, mBERT, DistilmBERT) dengan **tiga strategi fine-tuning**
(full fine-tuning, partial freezing, linear probing) pada **tiga seed acak**, sehingga seluruhnya berjumlah 27 run.
Setiap run dinilai dari akurasi, F1 makro, waktu latih, latensi inferensi, memori GPU puncak, dan jumlah parameter.
Hasil dilaporkan sebagai rata-rata ± simpangan baku antar-seed, bukan nilai terbaik dari satu kali percobaan.

Setiap bagian di bawah diawali penjelasan konsep, tujuan, dan klausul spesifikasi yang dipenuhi, lalu diikuti sel kode.

## Sel Rancangan (Soal 1 — Rancang Dulu Sebelum Ngoding)

Soal 1 pada spesifikasi mewajibkan rancangan ditulis **sebelum eksperimen pertama dijalankan**. Karena itu notebook menolak berjalan pada mode penuh bila bagian rancangan kosong (gerbang pada bagian 3).

### Matriks Eksperimen (3 Backbone x 3 Strategi x 3 Seed = 27 Run)

| ID | Backbone | Strategi Fine-Tuning | Variasi Random Seed |
|:---:|:---|:---|:---:|
| **E1** | `indobenchmark/indobert-base-p1` | Full Fine-Tuning | 42, 123, 2024 |
| **E2** | `indobenchmark/indobert-base-p1` | Partial Freezing (Freeze Emb + 6 Layer) | 42, 123, 2024 |
| **E3** | `indobenchmark/indobert-base-p1` | Linear Probing (Freeze 100% Backbone) | 42, 123, 2024 |
| **E4** | `bert-base-multilingual-cased` | Full Fine-Tuning | 42, 123, 2024 |
| **E5** | `bert-base-multilingual-cased` | Partial Freezing (Freeze Emb + 6 Layer) | 42, 123, 2024 |
| **E6** | `bert-base-multilingual-cased` | Linear Probing (Freeze 100% Backbone) | 42, 123, 2024 |
| **E7** | `distilbert-base-multilingual-cased` | Full Fine-Tuning | 42, 123, 2024 |
| **E8** | `distilbert-base-multilingual-cased` | Partial Freezing (Freeze Emb + 3 Layer) | 42, 123, 2024 |
| **E9** | `distilbert-base-multilingual-cased` | Linear Probing (Freeze 100% Backbone) | 42, 123, 2024 |

### Variabel Terkendali yang Dikunci
| Variabel | Nilai | Catatan |
|:---|:---|:---|
| `max_length` | 128 token | cukup untuk ulasan SmSA; teks lebih panjang dipotong |
| `batch_size` | 32 | sama untuk semua run |
| `num_epochs` | 4 | tetap, tanpa early stopping |
| Optimizer | AdamW (`weight_decay = 0.01`) | beta default PyTorch |
| LR schedule | linear warmup ratio 0.1 → linear decay | per strategi: full/partial 3e-5, probing 1e-3 |
| Gradient clipping | `max_grad_norm = 1.0` | — |
| Presisi | FP16 (mixed precision, AMP) | efisiensi GPU T4 |
| Seed | 42, 123, 2024 | sama untuk semua kombinasi |
| Himpunan evaluasi | `valid_preprocess.tsv` (pembagian resmi) | pelaporan Mean ± Std antar-seed |
| Benchmark latensi | batch = 1, 300 kalimat validasi pertama, FP32, satu sesi | mengikuti ketentuan: waktu inferensi hanya dibandingkan dalam satu sesi yang sama |

### Alasan Desain Singkat
- **Dataset SmSA dipilih** karena merupakan benchmark resmi klasifikasi sentimen Bahasa Indonesia
  (3 kelas) dan menjadi bagian dari IndoNLU, sehingga perbandingan backbone berbahasa Indonesia vs
  multibahasa dapat dilakukan secara adil pada domain yang sama.
- **Ukuran subset:** seluruh data latih resmi (tanpa subset) agar hasil tidak bergantung pada cuplikan acak.
- **3 seed per kombinasi:** untuk mengukur variasi acak antar-seed, sehingga klaim keunggulan sebuah
  backbone dapat dibandingkan terhadap besaran simpangan bakunya (bukan hanya satu kali coba).

### Hipotesis Penelitian
1. **H1 (Kesesuaian Bahasa Korpus):** IndoBERT mencapai F1 Macro dan Akurasi tertinggi secara
   signifikan dibandingkan mBERT dan DistilBERT pada mode Full Fine-Tuning, dengan selisih performa
   yang melampaui variasi acak antar-seed.
2. **H2 (Efisiensi vs Penurunan Performa DistilBERT):** DistilBERT Multilingual mempercepat waktu
   pelatihan dan latensi inferensi per kalimat >= 35% dibanding mBERT, dengan penurunan F1 Macro
   tidak melebihi 3.0 poin persentase.
3. **H3 (Pengaruh Pembekuan Lapisan):** F1 Macro mengikuti hierarki
   Full Fine-Tuning >= Partial Freezing > Linear Probing; partial freezing mempertahankan >= 96%
   performa full fine-tuning dengan penghematan waktu latih dan memori GPU >= 40%.

### Catatan Distribusi Kelas dan Metrik Utama
Distribusi kelas SmSA tidak seimbang (positive ~58%, negative ~31%, neutral ~11%). Karena kelas
`neutral` merupakan minoritas, **Macro F1-Score** dijadikan metrik utama penentu keunggulan model
(akurasi saja dapat memberi ilusi performa tinggi).

### Struktur Folder

```text
studio3_Deep-Learning_NLP/              <-- lokal (laptop)
├── notebooks/Studio_Proyek_3_NLP.ipynb
├── data/{train_preprocess.tsv, valid_preprocess.tsv}
├── hasil_eksperimen/                   <-- salinan berkas hasil dari Drive
└── laporan_dan_presentasi/

MyDrive/Studio3_NLP/                    <-- Google Drive (dibuat otomatis oleh notebook)
├── data/
└── output/
    ├── hasil_smsa_<kode>.csv           (satu baris per run)
    ├── rekap_agregat_smsa_<kode>.csv   (rata-rata ± std per kombinasi)
    ├── prediksi_smsa_<kode>.csv        (prediksi tiap run, untuk confusion matrix)
    ├── latensi_sesi_<kode>.csv         (benchmark latensi satu sesi)
    ├── evaluasi_hipotesis_<kode>.csv
    ├── grafik_*.png
    └── model_terbaik/
```

## 1. Instalasi Pustaka

**Klausul spesifikasi (Hal-hal Teknis):** platform Google Colab atau Kaggle dengan GPU gratis; pustaka wajib
HuggingFace `transformers` dan `scikit-learn`.

Dataset SmSA diunduh langsung dari repositori resmi IndoNLU, sehingga pustaka `datasets` tidak diperlukan
(pustaka itu hanya relevan bila memakai IMDb). `scipy` dipakai untuk uji statistik pada evaluasi hipotesis, sedangkan
`pandas`, `matplotlib`, dan `seaborn` dipakai untuk pengolahan tabel dan visualisasi. Colab sudah memuat sebagian besar
pustaka ini; opsi `-q` menyenyapkan keluaran, dan batas versi minimum menjamin API yang dipakai tersedia.

In [ ]:
%pip install -q "transformers>=4.40" scikit-learn pandas matplotlib seaborn scipy

## 2. Setup Lingkungan, Google Drive, dan Perangkat Komputasi

**Tujuan:** menyiapkan impor pustaka, lokasi penyimpanan yang tahan terhadap putusnya sesi, dan deteksi GPU.

- **Google Drive:** sesi Colab gratis dapat terputus kapan saja dan memori sesinya hilang. Seluruh data, log hasil, grafik,
  dan satu model terbaik karenanya ditulis ke `MyDrive/Studio3_NLP/`, sehingga eksperimen dapat dilanjutkan tanpa mengulang
  dari awal. Bila notebook dijalankan di luar Colab (laptop atau Kaggle), folder kerja memakai direktori proyek lokal.
- **Identitas sesi:** setiap eksekusi notebook diberi `SESI_ID` dan nama GPU dicatat. Ketentuan spesifikasi menyatakan waktu
  inferensi hanya boleh dibandingkan dalam satu sesi yang sama; kedua catatan ini membuat hal tersebut dapat diaudit.

In [ ]:
import os, gc, time, json, hashlib, random, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Studio3_NLP')
    IN_COLAB = True
    print("Mode: Google Colab (Drive terhubung)")
except ImportError:
    # notebook berada di <proyek>/notebooks/, jadi akar proyek satu level di atasnya
    BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    IN_COLAB = False
    print(f"Mode: lokal / Kaggle (BASE_DIR = {BASE_DIR})")

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
SESI_ID = time.strftime("%Y%m%d-%H%M%S")
print(f"Direktori data   : {DATA_DIR}\nDirektori output : {OUTPUT_DIR}")
print(f"Perangkat        : {DEVICE} ({GPU_NAME}) | sesi {SESI_ID}")

## 3. Seed, Konfigurasi Terkunci, dan Gerbang Rancangan

**Klausul spesifikasi:**
- *Soal 1* — variabel `max_length`, batch size, jumlah epoch, dan minimal 3 seed acak yang sama untuk semua run dikunci sejak awal.
- *Reproduksibilitas* — `set_seed()` wajib dipanggil agar kode dapat dijalankan ulang dengan hasil konsisten.
- *Learning rate* — mengikuti strategi: 2e-5 s.d. 5e-5 untuk full/partial, 1e-3 s.d. 5e-3 untuk linear probing (head dilatih dari bobot
  acak sehingga butuh langkah yang jauh lebih besar). Nilai yang dipilih: 3e-5 dan 1e-3.
- *Nama berkas hasil* — `hasil_<dataset>_<kode>.csv` dengan kode yang berubah bila konfigurasi berubah. Kode di sini adalah
  hash 6 karakter dari seluruh konfigurasi, sehingga hasil dari konfigurasi berbeda tidak pernah tercampur.

**Mode eksekusi.** `MODE = "smoke"` menjalankan uji cepat (subset kecil, 1 seed, 1 epoch) ke folder terpisah untuk memastikan
tidak ada galat sebelum eksperimen panjang. `MODE = "full"` menjalankan matriks penuh dan **menolak berjalan bila rancangan kosong**,
sesuai ketentuan Soal 1.

In [ ]:
def set_seed(seed: int = 42):
    """Menyeragamkan sumber keacakan Python, NumPy, dan PyTorch (CPU + GPU)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

MODE = "full"   # "smoke" untuk uji cepat, "full" untuk 27 run

CONFIG = {
    "versi_kode": 2,   # dinaikkan saat logika pencatatan berubah, agar hasil lama tidak tercampur
    "dataset_name": "SmSA_IndoNLU",
    "num_classes": 3,
    "label_map": {"negative": 0, "neutral": 1, "positive": 2},
    "inv_label_map": {0: "negative", 1: "neutral", 2: "positive"},
    "max_length": 128,
    "batch_size": 32,
    "num_epochs": 4,
    "lr_full": 3e-5,
    "lr_partial": 3e-5,
    "lr_probing": 1e-3,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,
    "seeds": [42, 123, 2024],
    "backbones": {
        "indobert": "indobenchmark/indobert-base-p1",
        "mbert": "bert-base-multilingual-cased",
        "distilmbert": "distilbert-base-multilingual-cased",
    },
    "strategies": ["full", "partial", "probing"],
    "partial_frozen_layers": {"indobert": 6, "mbert": 6, "distilmbert": 3},   # separuh lapis awal
    "latency_samples": 300,
    "latency_warmup": 10,
    "latency_repeats": 3,
}

if MODE == "smoke":
    CONFIG.update({"seeds": [42], "num_epochs": 1, "smoke_train_n": 256, "smoke_valid_n": 128,
                   "latency_samples": 40, "latency_warmup": 5, "latency_repeats": 1})

_hash_input = {k: CONFIG[k] for k in sorted(CONFIG) if k != "inv_label_map"}
KODE = hashlib.sha1(json.dumps(_hash_input, sort_keys=True).encode()).hexdigest()[:6]
print(f"MODE = {MODE} | kode konfigurasi = {KODE}")

# Ringkasan terstruktur dari sel Rancangan di atas; diperiksa oleh gerbang di bawahnya
RANCANGAN = {
    "matriks": "3 backbone (IndoBERT, mBERT, DistilmBERT) x 3 strategi (full, partial, probing) x 3 seed = 27 run",
    "H1": "IndoBERT unggul pada F1 makro dibanding mBERT dan DistilmBERT (full FT), melampaui std antar-seed.",
    "H2": "DistilmBERT >=35% lebih cepat (latih & latensi) dibanding mBERT dengan penurunan F1 makro <=3 poin.",
    "H3": "F1: full >= partial > probing; partial >=96% F1 full dengan hemat waktu & memori >=40%.",
    "alasan_desain": "SmSA: benchmark sentimen Indonesia 3 kelas; seluruh data latih resmi dipakai; 3 seed untuk std.",
    "variabel_terkunci": {k: CONFIG[k] for k in ("max_length", "batch_size", "num_epochs", "seeds")},
}
if MODE == "full":
    kosong = [k for k, v in RANCANGAN.items() if not v]
    assert not kosong, f"Mode penuh DITOLAK: bagian Rancangan masih kosong -> {kosong}"
    assert len(CONFIG["seeds"]) >= 3 and len(CONFIG["backbones"]) >= 3, "Minimal 3 seed dan 3 backbone."
    print("Gerbang Rancangan lolos.")

## 4. Pengunduhan dan Pembersihan Data SmSA

**Klausul spesifikasi (Pembagian & pembersihan data):** untuk SmSA dipakai pembagian resmi; notebook membuang duplikat,
teks berlabel bertentangan, dan teks validasi yang ada di data latih.

Tiga pemeriksaan kualitas dilakukan **dengan urutan yang disengaja**:
1. **Label konflik** (teks sama, label berbeda) dideteksi lebih dulu. Bila deduplikasi dilakukan lebih dulu, tiap teks hanya
   tersisa satu label dan konflik tidak mungkin terdeteksi lagi.
2. **Duplikat persis** (teks dan label sama) dikurangi menjadi satu salinan agar contoh yang berulang tidak berbobot ganda.
3. **Kebocoran data:** teks latih yang juga muncul di himpunan validasi dibuang, supaya skor validasi tidak menggelembung
   karena model pernah melihat jawabannya.

Label teks dipetakan ke bilangan bulat, dan sel ini memastikan tidak ada label di luar tiga kelas. Distribusi kelas dicetak karena
SmSA tidak seimbang (kelas *neutral* minoritas), yang menjadi alasan **F1 makro** dijadikan metrik utama dan bukan akurasi.
Label uji resmi SmSA disembunyikan, sehingga evaluasi memakai himpunan validasi resmi.

In [ ]:
if MODE == "smoke":
    OUTPUT_DIR = OUTPUT_DIR / "smoke_test"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_URL = "https://raw.githubusercontent.com/indobenchmark/indonlu/master/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv"
VALID_URL = "https://raw.githubusercontent.com/indobenchmark/indonlu/master/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv"

train_file = DATA_DIR / "train_preprocess.tsv"
valid_file = DATA_DIR / "valid_preprocess.tsv"
for url, f in ((TRAIN_URL, train_file), (VALID_URL, valid_file)):
    if not f.exists():
        print(f"Mengunduh {f.name} ...")
        urllib.request.urlretrieve(url, f)

df_train_raw = pd.read_csv(train_file, sep='\t', names=['text', 'label'], header=None)
df_valid_raw = pd.read_csv(valid_file, sep='\t', names=['text', 'label'], header=None)
n_mentah = len(df_train_raw)

# 1. label konflik (harus sebelum deduplikasi)
n_label = df_train_raw.groupby('text')['label'].nunique()
teks_konflik = set(n_label[n_label > 1].index)
df_train = df_train_raw[~df_train_raw['text'].isin(teks_konflik)]
n_konflik_baris = n_mentah - len(df_train)

# 2. duplikat persis
sebelum = len(df_train)
df_train = df_train.drop_duplicates(subset=['text'])
n_duplikat = sebelum - len(df_train)

# 3. kebocoran ke validasi
teks_valid = set(df_valid_raw['text'])
sebelum = len(df_train)
df_train = df_train[~df_train['text'].isin(teks_valid)].copy()
n_bocor = sebelum - len(df_train)

df_train['label_id'] = df_train['label'].map(CONFIG['label_map'])
df_valid = df_valid_raw.copy()
df_valid['label_id'] = df_valid['label'].map(CONFIG['label_map'])
assert df_train['label_id'].notna().all() and df_valid['label_id'].notna().all(), "Ada label di luar label_map!"
df_train['label_id'] = df_train['label_id'].astype(int)
df_valid['label_id'] = df_valid['label_id'].astype(int)

print(f"Mentah={n_mentah} | baris konflik dibuang={n_konflik_baris} ({len(teks_konflik)} teks) | "
      f"duplikat dibuang={n_duplikat} | bocor ke validasi dibuang={n_bocor}")
print(f"Data bersih: train={len(df_train)}, valid={len(df_valid)}")
print("Distribusi kelas data latih (%):")
print((df_train['label'].value_counts(normalize=True) * 100).round(2))

if MODE == "smoke":
    df_train = df_train.sample(CONFIG["smoke_train_n"], random_state=0).reset_index(drop=True)
    df_valid = df_valid.sample(CONFIG["smoke_valid_n"], random_state=0).reset_index(drop=True)
    print(f"[SMOKE] subset: train={len(df_train)}, valid={len(df_valid)}")

## 5. Dataset PyTorch dengan Padding Dinamis

**Tujuan:** mengubah teks menjadi tensor yang dapat dimakan model, secara efisien.

Setiap model memiliki tokenizer sendiri (kosakata WordPiece yang berbeda), sehingga tokenisasi dilakukan per backbone.
Dua keputusan efisiensi diambil:
- Teks ditokenisasi **sekali** di awal, bukan berulang pada setiap iterasi pelatihan.
- Padding dilakukan **per batch** (sepanjang kalimat terpanjang di batch itu, maksimal `max_length = 128`), bukan memaksa semua
  kalimat menjadi 128 token. Kalimat SmSA umumnya jauh lebih pendek, sehingga komputasi pada token *padding* yang sia-sia berkurang
  banyak dan waktu latih lebih mencerminkan beban kerja sebenarnya. Aturan ini berlaku sama untuk seluruh run, jadi perbandingan tetap adil.

Fungsi `hitung_parameter` menghitung jumlah parameter total dan yang dilatih; keduanya wajib tercatat pada tabel hasil (Soal 2).

In [ ]:
class SmSADataset(Dataset):
    """Tokenisasi sekali di awal; padding dilakukan per batch oleh collator."""
    def __init__(self, texts, labels, tokenizer, max_len=128):
        enc = tokenizer(list(map(str, texts)), truncation=True, max_length=max_len,
                        padding=False, return_attention_mask=True)
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx],
                'attention_mask': self.attention_mask[idx],
                'label': self.labels[idx]}

def buat_collator(tokenizer):
    def collate(batch):
        labels = torch.tensor([b['label'] for b in batch], dtype=torch.long)
        padded = tokenizer.pad(
            [{'input_ids': b['input_ids'], 'attention_mask': b['attention_mask']} for b in batch],
            padding=True, return_tensors='pt')
        return {'input_ids': padded['input_ids'],
                'attention_mask': padded['attention_mask'],
                'label': labels}
    return collate

def hitung_parameter(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

## 6. Strategi Fine-Tuning dan Logika Pembekuan Lapis

**Klausul spesifikasi (Soal 3):** head replacement wajib untuk semua run; full fine-tuning wajib sebagai baseline;
linear probing atau partial freezing dipilih minimal satu — notebook ini mengerjakan **keduanya**. Kode pembekuan harus terlihat
jelas di notebook, bukan hanya disebut di teks.

| Strategi | Parameter yang dilatih | Pembekuan |
|:---|:---|:---|
| **Full fine-tuning** | seluruh parameter | tidak ada |
| **Partial freezing** | separuh lapis atas + pooler + head | embedding dan separuh lapis awal (12 → 6 untuk BERT; 6 → 3 untuk DistilBERT) |
| **Linear probing** | hanya head klasifikasi | seluruh backbone |

**Head replacement.** `AutoModelForSequenceClassification(..., num_labels=3)` membuang head pra-latih lama dan memasang head baru berbobot
acak untuk 3 kelas. Pesan *MISSING* (head baru) dan *UNEXPECTED* (head pra-latih yang dibuang) saat memuat model adalah perilaku yang diharapkan.
Pada DistilBERT, head terdiri dari dua lapis (`pre_classifier` dan `classifier`); keduanya harus ikut dilatih pada linear probing.

**Verifikasi, bukan asumsi.** Setelah mengatur `requires_grad`, fungsi memeriksa dengan `assert` bahwa bagian yang seharusnya beku
benar-benar beku dan sisanya aktif. Sesuai spesifikasi, jumlah parameter yang dilatih harus berurutan **probing < partial < full**;
urutan ini diperiksa lagi pada rekapitulasi hasil.

In [ ]:
def _komponen_backbone(model):
    """Mengembalikan (embeddings, daftar lapis encoder, daftar modul head) untuk BERT / DistilBERT."""
    if hasattr(model, 'bert'):
        return model.bert.embeddings, list(model.bert.encoder.layer), [model.classifier]
    if hasattr(model, 'distilbert'):
        return (model.distilbert.embeddings, list(model.distilbert.transformer.layer),
                [model.pre_classifier, model.classifier])
    raise ValueError("Arsitektur tidak dikenali (hanya BERT / DistilBERT).")

def atur_strategi_fine_tuning(model, backbone_key: str, strategi: str):
    """Mengatur requires_grad sesuai strategi lalu memverifikasinya."""
    emb, layers, head = _komponen_backbone(model)
    n_layer = len(layers)

    for p in model.parameters():          # titik awal seragam: semua dilatih
        p.requires_grad = True

    if strategi == "full":
        pass

    elif strategi == "partial":
        n_beku = CONFIG['partial_frozen_layers'][backbone_key]
        assert n_beku == n_layer // 2, f"{backbone_key}: {n_beku} bukan separuh dari {n_layer} lapis"
        for p in emb.parameters():
            p.requires_grad = False
        for layer in layers[:n_beku]:
            for p in layer.parameters():
                p.requires_grad = False

    elif strategi == "probing":
        for p in model.parameters():
            p.requires_grad = False
        for m in head:
            for p in m.parameters():
                p.requires_grad = True
    else:
        raise ValueError(strategi)

    if strategi == "partial":
        beku = all(not p.requires_grad for p in emb.parameters()) and \
               all(not p.requires_grad for l in layers[:n_beku] for p in l.parameters())
        aktif = all(p.requires_grad for l in layers[n_beku:] for p in l.parameters()) and \
                all(p.requires_grad for m in head for p in m.parameters())
        assert beku and aktif, "Freezing partial tidak sesuai definisi."
    if strategi == "probing":
        id_head = {id(p) for m in head for p in m.parameters()}
        assert all(p.requires_grad == (id(p) in id_head) for p in model.parameters()), \
            "Probing: hanya head yang boleh dilatih."

    tot, train = hitung_parameter(model)
    print(f"[{backbone_key.upper()} - {strategi.upper()}] total {tot:,} | dilatih {train:,} ({train/tot*100:.2f}%) "
          f"| lapis={n_layer} | head={[type(m).__name__ for m in head]}")
    return model

## 7. Fungsi Pelatihan, Evaluasi, dan Pengukuran Metrik

**Klausul spesifikasi (Metrik per run, otomatis):** akurasi, F1 makro, waktu latih, waktu inferensi (ms/kalimat),
parameter total dan yang dilatih, serta memori puncak.

Komponen utama:
- **Presisi campuran (FP16)** lewat `autocast` dan `GradScaler` menghemat memori dan waktu pada GPU T4. *Gradient clipping* (norma maks 1,0)
  menjaga pelatihan tetap stabil.
- **Optimizer AdamW** dengan *warmup* linear 10% lalu penurunan linear, hanya pada parameter yang `requires_grad`.
- **Waktu latih** diukur dengan `perf_counter` setelah `cuda.synchronize()`, karena operasi GPU berjalan asinkron dan tanpa sinkronisasi
  waktu akan terukur terlalu kecil.
- **Memori puncak** memakai `max_memory_allocated`, dengan puncak di-reset *setelah* bobot model berada di GPU sehingga ukuran model ikut terhitung. Fungsi memperingatkan
  bila ada sisa memori dari run sebelumnya, yang akan menggelembungkan angka.
- **Latensi inferensi** adalah rata-rata waktu per **satu kalimat** (batch = 1, tanpa padding, FP32) pada 300 kalimat validasi pertama yang sama untuk semua model,
  setelah pemanasan yang tidak dihitung.
- Prediksi pada seluruh himpunan validasi dikembalikan agar *confusion matrix* dapat dibuat kemudian tanpa melatih ulang.

In [ ]:
USE_AMP = torch.cuda.is_available()   # FP16 hanya bila ada GPU

def ukur_latensi_inferensi(model, dataset, device, n_sample, n_warmup):
    """Latensi per kalimat (ms): batch=1, tanpa padding, FP32, kalimat yang sama untuk semua model."""
    model.eval()
    cuda = device.type == "cuda"
    waktu = []
    n_total = min(n_sample + n_warmup, len(dataset))
    with torch.inference_mode():
        for i in range(n_total):
            ids = torch.tensor([dataset.input_ids[i]], device=device)
            msk = torch.tensor([dataset.attention_mask[i]], device=device)
            if cuda: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(input_ids=ids, attention_mask=msk)
            if cuda: torch.cuda.synchronize()
            t1 = time.perf_counter()
            if i >= n_warmup:
                waktu.append(t1 - t0)
    return float(np.mean(waktu) * 1000.0)

def latih_satu_run(model, train_loader, val_loader, val_dataset, optimizer, scheduler, epochs, device):
    criterion = nn.CrossEntropyLoss()
    cuda = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    if cuda:
        torch.cuda.synchronize()
        base_mb = torch.cuda.memory_allocated(device) / 1024**2
        if base_mb > 50:
            print(f"[PERINGATAN] Memori GPU sebelum run sudah {base_mb:.0f} MB (sisa run sebelumnya?).")
    model.to(device)
    if cuda:
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize()

    t_mulai = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits.float(), labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
    if cuda: torch.cuda.synchronize()
    waktu_latih_detik = time.perf_counter() - t_mulai
    memori_puncak_mb = torch.cuda.max_memory_allocated(device) / 1024**2 if cuda else 0.0

    model.eval()
    semua_pred, semua_label = [], []
    with torch.inference_mode():
        for batch in val_loader:
            logits = model(input_ids=batch['input_ids'].to(device),
                           attention_mask=batch['attention_mask'].to(device)).logits
            semua_pred.extend(torch.argmax(logits, dim=1).cpu().tolist())
            semua_label.extend(batch['label'].tolist())

    f1_kelas = f1_score(semua_label, semua_pred, average=None, labels=[0, 1, 2])
    latensi_ms = ukur_latensi_inferensi(model, val_dataset, device,
                                        CONFIG['latency_samples'], CONFIG['latency_warmup'])
    return {
        "accuracy": accuracy_score(semua_label, semua_pred),
        "f1_macro": f1_score(semua_label, semua_pred, average='macro'),
        "f1_negative": f1_kelas[0], "f1_neutral": f1_kelas[1], "f1_positive": f1_kelas[2],
        "train_time_s": waktu_latih_detik,
        "latency_ms": latensi_ms,
        "peak_memory_mb": memori_puncak_mb,
        "pred": semua_pred, "label": semua_label,
    }

## 8. Eksekusi Matriks Eksperimen (27 Run) dengan Penyimpanan Otomatis

**Klausul spesifikasi:** protokol dan hiperparameter identik untuk semua backbone (Soal 2); tiap kombinasi dijalankan dengan 3 seed berbeda;
`set_seed()` dipanggil sebelum setiap run; satu backbone dijalankan dengan semua strategi agar perbandingan apple-to-apple (Soal 3).
Berkas hasil dibuat otomatis oleh notebook. Spesifikasi memperkirakan 3–5 menit per run pada T4, sehingga 27 run memerlukan sekitar 1,5–2 jam.

Mekanisme penting pada sel ini:
- **Urutan seed.** `set_seed` dipanggil lebih dulu, lalu head klasifikasi baru dibuat, sehingga inisialisasi head ditentukan seed. Urutan *shuffle* data
  memakai `Generator` yang dikunci seed, bukan keadaan acak global.
- **Ketahanan terhadap putus sesi.** Setiap run selesai langsung ditulis ke Drive (CSV hasil dan CSV prediksi) memakai penulisan atomik: ke berkas
  sementara dahulu lalu diganti, sehingga berkas tidak rusak bila sesi terputus saat menulis. Run yang sudah tercatat dilewati saat notebook dijalankan ulang.
- **Model terbaik.** Hanya satu model dengan F1 makro tertinggi disimpan, untuk menghemat kuota Drive. Skor terbaik diwarisi dari riwayat sehingga run lanjutan
  tidak menimpanya dengan model yang lebih buruk.
- **Pembersihan memori.** Model, optimizer, scheduler, dan loader dihapus lalu `empty_cache()` dipanggil di antara run agar memori puncak tiap run tidak saling mencemari.

In [ ]:
berkas_csv_hasil = OUTPUT_DIR / f"hasil_smsa_{KODE}.csv"
berkas_prediksi = OUTPUT_DIR / f"prediksi_smsa_{KODE}.csv"
best_model_path = OUTPUT_DIR / "model_terbaik"
KUNCI = ["backbone", "strategy", "seed"]

def simpan_csv_atomik(df, tujuan: Path):
    """Tulis ke berkas sementara lalu ganti, agar CSV tidak rusak bila sesi putus saat menulis."""
    tmp = tujuan.with_suffix(".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, tujuan)

hasil_semua_run = pd.read_csv(berkas_csv_hasil).to_dict('records') if berkas_csv_hasil.exists() else []
pred_frames = [pd.read_csv(berkas_prediksi)] if berkas_prediksi.exists() else []

def _kunci_selesai():
    k_hasil = {(r['backbone'], r['strategy'], r['seed']) for r in hasil_semua_run}
    if not pred_frames:
        return set()
    p = pd.concat(pred_frames)[KUNCI].drop_duplicates()
    k_pred = set(map(tuple, p.values.tolist()))
    return k_hasil & k_pred   # run dianggap selesai bila hasil DAN prediksinya tersimpan

selesai = _kunci_selesai()
hasil_semua_run = [r for r in hasil_semua_run if (r['backbone'], r['strategy'], r['seed']) in selesai]
if selesai:
    print(f"Melanjutkan eksperimen: {len(selesai)} run sudah selesai (kode {KODE}).")

best_f1_global = max((r['f1_macro'] for r in hasil_semua_run), default=-1.0)
if best_f1_global > 0 and not best_model_path.exists():
    best_f1_global = -1.0   # berkas model terbaik lama hilang -> simpan ulang pada run berikutnya

for bb_key, bb_path in CONFIG["backbones"].items():
    print(f"\n{'='*65}\nBACKBONE: {bb_key.upper()} ({bb_path})\n{'='*65}")
    tokenizer = AutoTokenizer.from_pretrained(bb_path)
    collate = buat_collator(tokenizer)

    train_dataset = SmSADataset(df_train['text'], df_train['label_id'], tokenizer, CONFIG['max_length'])
    val_dataset = SmSADataset(df_valid['text'], df_valid['label_id'], tokenizer, CONFIG['max_length'])
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate)

    for strat in CONFIG["strategies"]:
        for seed in CONFIG["seeds"]:
            if (bb_key, strat, seed) in selesai:
                print(f"[SKIP] {bb_key} | {strat} | seed {seed} sudah tercatat.")
                continue

            print(f"\n---> {bb_key} | {strat} | seed {seed}")
            set_seed(seed)
            g = torch.Generator().manual_seed(seed)
            train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
                                      collate_fn=collate, generator=g)

            # Head replacement: head baru (3 kelas) berbobot acak, dibuat setelah set_seed
            model = AutoModelForSequenceClassification.from_pretrained(bb_path, num_labels=CONFIG['num_classes'])
            assert model.classifier.out_features == CONFIG['num_classes']
            model = atur_strategi_fine_tuning(model, bb_key, strat)
            total_param, trainable_param = hitung_parameter(model)

            lr = {"full": CONFIG['lr_full'], "partial": CONFIG['lr_partial'], "probing": CONFIG['lr_probing']}[strat]
            optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                          lr=lr, weight_decay=CONFIG['weight_decay'])
            total_steps = len(train_loader) * CONFIG['num_epochs']
            scheduler = get_linear_schedule_with_warmup(
                optimizer, num_warmup_steps=int(total_steps * CONFIG['warmup_ratio']),
                num_training_steps=total_steps)

            res = latih_satu_run(model, train_loader, val_loader, val_dataset,
                                 optimizer, scheduler, CONFIG['num_epochs'], DEVICE)

            # prediksi disimpan lebih dulu; baris hasil menyusul (run sah bila keduanya ada)
            pred_frames.append(pd.DataFrame({
                "backbone": bb_key, "strategy": strat, "seed": seed,
                "idx": range(len(res["label"])), "y_true": res["label"], "y_pred": res["pred"]}))
            simpan_csv_atomik(pd.concat(pred_frames).drop_duplicates(KUNCI + ["idx"], keep="last"), berkas_prediksi)

            catatan = {"backbone": bb_key, "strategy": strat, "seed": seed, "lr": lr,
                       **{k: v for k, v in res.items() if k not in ("pred", "label")},
                       "total_params": total_param, "trainable_params": trainable_param,
                       "gpu": GPU_NAME, "sesi": SESI_ID}
            hasil_semua_run.append(catatan)
            simpan_csv_atomik(pd.DataFrame(hasil_semua_run), berkas_csv_hasil)
            print(f"Tersimpan -> F1={res['f1_macro']:.4f} Acc={res['accuracy']:.4f} "
                  f"Latih={res['train_time_s']:.0f}s Mem={res['peak_memory_mb']:.0f}MB")

            if res["f1_macro"] > best_f1_global:
                best_f1_global = res["f1_macro"]
                model.save_pretrained(best_model_path)
                tokenizer.save_pretrained(best_model_path)
                print(f"Model terbaik baru ({bb_key}-{strat}-seed{seed}) disimpan.")

            del model, optimizer, scheduler, train_loader, res
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    del train_dataset, val_dataset, val_loader
    gc.collect()

print("\nSeluruh eksperimen selesai.")

## 9. Benchmark Latensi dalam Satu Sesi

**Klausul spesifikasi:** *waktu inferensi hanya dibandingkan dalam satu sesi yang sama.* Karena 27 run dapat tersebar pada beberapa sesi Colab
(dengan GPU, suhu, dan beban berbeda), latensi yang dicatat di tiap run tidak dapat dibandingkan langsung antar sesi.

Latensi suatu model ditentukan oleh **arsitekturnya** (jumlah lapis, ukuran tersembunyi), bukan oleh strategi pelatihan atau bobot hasil fine-tuning:
forward pass-nya sama persis. Karena itu latensi dibandingkan lewat benchmark terpisah yang memuat ketiga arsitektur **berurutan dalam satu sesi** ini, dengan pengulangan
bergantian antar backbone (agar pergeseran suhu atau beban GPU tidak menguntungkan satu model) dan nilai dilaporkan sebagai rata-rata ± std antar-pengulangan.
Tabel ini menjadi dasar pengujian H2 pada bagian evaluasi hipotesis. Latensi per run pada CSV hasil tetap disimpan sebagai informasi pelengkap.

In [ ]:
LAT_DATASET = {}
for bb_key, bb_path in CONFIG["backbones"].items():
    tok = AutoTokenizer.from_pretrained(bb_path)
    LAT_DATASET[bb_key] = SmSADataset(df_valid['text'], df_valid['label_id'], tok, CONFIG['max_length'])

baris_lat = []
for ulang in range(CONFIG["latency_repeats"]):          # bergantian antar-backbone di setiap putaran
    for bb_key, bb_path in CONFIG["backbones"].items():
        set_seed(ulang)
        m = AutoModelForSequenceClassification.from_pretrained(bb_path, num_labels=CONFIG['num_classes']).to(DEVICE)
        lat = ukur_latensi_inferensi(m, LAT_DATASET[bb_key], DEVICE, CONFIG['latency_samples'], CONFIG['latency_warmup'])
        baris_lat.append({"backbone": bb_key, "ulang": ulang, "latency_ms": lat, "gpu": GPU_NAME, "sesi": SESI_ID})
        del m
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df_lat = pd.DataFrame(baris_lat)
simpan_csv_atomik(df_lat, OUTPUT_DIR / f"latensi_sesi_{KODE}.csv")
ringkas_lat = df_lat.groupby("backbone").latency_ms.agg(["mean", "std"]).reindex(list(CONFIG["backbones"]))
display(ringkas_lat.round(3).rename(columns={"mean": "ms/kalimat (mean)", "std": "std antar-pengulangan"}))

## 10. Rekapitulasi Agregat: Rata-rata ± Simpangan Baku

**Klausul spesifikasi (Pelaporan hasil):** rata-rata ± simpangan baku antar-seed, **bukan** hanya nilai terbaik dari satu kali coba.
Berkas `rekap_agregat_smsa_<kode>.csv` dibuat otomatis.

Simpangan baku memakai penyebut *n − 1* (`ddof = 1`, sampel) karena tiga seed adalah cuplikan dari kemungkinan seed yang tak terhingga.
Sebelum menghitung rekap, sel ini memvalidasi dua hal: (1) seluruh 27 run lengkap, dan (2) jumlah parameter yang dilatih berurutan
**probing < partial < full** di setiap backbone — pemeriksaan yang diminta spesifikasi pada definisi partial freezing. Kolom latensi
berasal dari benchmark satu sesi pada bagian 9.

In [ ]:
df_hasil = pd.DataFrame(hasil_semua_run)

n_harapan = len(CONFIG["backbones"]) * len(CONFIG["strategies"]) * len(CONFIG["seeds"])
assert len(df_hasil) == n_harapan, f"Run belum lengkap: {len(df_hasil)}/{n_harapan}"
for bb, g in df_hasil.groupby("backbone"):
    tp = g.groupby("strategy")["trainable_params"].first()
    assert tp["probing"] < tp["partial"] < tp["full"], f"Hierarki parameter salah pada {bb}: {tp.to_dict()}"
print("Validasi lolos: 27 run lengkap dan parameter dilatih probing < partial < full.")
print(f"Run berasal dari {df_hasil['sesi'].nunique()} sesi; GPU: {', '.join(sorted(df_hasil['gpu'].unique()))}")

metrik = ["accuracy", "f1_macro", "f1_neutral", "train_time_s", "peak_memory_mb"]
rekap_agregat = df_hasil.groupby(["backbone", "strategy"]).agg(
    n_seed=("seed", "nunique"),
    total_params=("total_params", "first"),
    trainable_params=("trainable_params", "first"),
    **{f"{m}_{s}": (m, s) for m in metrik for s in ("mean", "std")}
).reset_index()

lat_bb = df_lat.groupby("backbone").latency_ms.agg(latency_ms_mean="mean", latency_ms_std="std").reset_index()
rekap_agregat = rekap_agregat.merge(lat_bb, on="backbone", how="left")

semua_metrik = metrik + ["latency_ms"]
for m in semua_metrik:
    rekap_agregat[f"{m}_ringkas"] = rekap_agregat.apply(
        lambda r: f"{r[m+'_mean']:.4f} ± {r[m+'_std']:.4f}" if m.startswith(("acc", "f1")) else
                  f"{r[m+'_mean']:.2f} ± {r[m+'_std']:.2f}", axis=1)

berkas_rekap = OUTPUT_DIR / f"rekap_agregat_smsa_{KODE}.csv"
simpan_csv_atomik(rekap_agregat, berkas_rekap)
print(f"Rekap agregat tersimpan di {berkas_rekap}")
display(rekap_agregat[["backbone", "strategy", "n_seed", "trainable_params"] +
                      [f"{m}_ringkas" for m in semua_metrik]])

## 11. Visualisasi Perbandingan Performa dan Biaya

Empat gambar diringkas dari hasil agregat. Pada semua gambar, **batang = rata-rata antar-seed, garis hitam = simpangan baku, titik putih = nilai tiap seed**,
sehingga sebaran antar-seed terlihat langsung. Warna strategi konsisten di seluruh gambar.

1. **F1 makro dan akurasi** — mengukur kualitas model. Sumbu *y* dipotong agar selisih antar-model terbaca, dan hal ini dinyatakan pada judul.
2. **Waktu latih dan memori GPU puncak** — biaya komputasi yang disebut spesifikasi sebagai pertimbangan praktis.
3. **Trade-off akurasi vs latensi** — latensi dari benchmark satu sesi (bagian 9). Titik di kiri atas paling menguntungkan. Titik berwarna sama dalam satu backbone
   memiliki latensi yang sama karena arsitekturnya sama.
4. **F1 per kelas** — memperlihatkan kelas *neutral* (minoritas) sebagai kelas tersulit.

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11.5,
    "xtick.labelsize": 11, "ytick.labelsize": 10.5, "legend.fontsize": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
    "grid.color": "#dddddd", "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
})

NAMA_BB = {"indobert": "IndoBERT", "mbert": "mBERT", "distilmbert": "DistilmBERT"}
KEY_BB = {v: k for k, v in NAMA_BB.items()}
NAMA_ST = {"full": "Full FT", "partial": "Partial freezing", "probing": "Linear probing"}
WARNA = {"Full FT": "#1b5e8c", "Partial freezing": "#e08a1e", "Linear probing": "#7a7a7a"}
URUT_BB = [NAMA_BB[k] for k in CONFIG["backbones"]]
URUT_ST = [NAMA_ST[s] for s in CONFIG["strategies"]]

d = df_hasil.copy()
d["Backbone"] = d["backbone"].map(NAMA_BB)
d["Strategi"] = d["strategy"].map(NAMA_ST)
agg = d.groupby(["Backbone", "Strategi"]).agg(
    f1=("f1_macro", "mean"), f1_sd=("f1_macro", "std"),
    acc=("accuracy", "mean"), acc_sd=("accuracy", "std"),
    waktu=("train_time_s", "mean"), waktu_sd=("train_time_s", "std"),
    mem=("peak_memory_mb", "mean"), mem_sd=("peak_memory_mb", "std")).reset_index().fillna(0)

def batang_dengan_seed(ax, kolom, kolom_sd, kolom_raw, judul, ylabel, fmt):
    """Batang = rata-rata, garis = std antar-seed, titik = nilai tiap seed."""
    lebar = 0.26
    for j, st in enumerate(URUT_ST):
        for i, bb in enumerate(URUT_BB):
            baris = agg[(agg.Backbone == bb) & (agg.Strategi == st)].iloc[0]
            x = i + (j - 1) * lebar
            ax.bar(x, baris[kolom], lebar * 0.92, color=WARNA[st], label=st if i == 0 else None, zorder=2)
            ax.errorbar(x, baris[kolom], yerr=baris[kolom_sd], color="black", capsize=3, lw=1, zorder=4)
            titik = d[(d.Backbone == bb) & (d.Strategi == st)][kolom_raw]
            ax.scatter([x] * len(titik), titik, s=16, color="white", edgecolor="black", lw=0.7, zorder=5)
            ax.annotate(fmt.format(baris[kolom]), (x, baris[kolom] + baris[kolom_sd]),
                        xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=9, zorder=6)
    ax.set_xticks(range(len(URUT_BB)))
    ax.set_xticklabels(URUT_BB)
    ax.set_title(judul, loc="left", pad=12)
    ax.set_ylabel(ylabel)
    ax.grid(axis="x", visible=False)
    ax.margins(y=0.12)

# Gambar 1: F1 makro dan akurasi
fig1, ax = plt.subplots(1, 2, figsize=(15, 5.2), constrained_layout=True)
batang_dengan_seed(ax[0], "f1", "f1_sd", "f1_macro", "F1 makro (sumbu y dipotong)", "F1 makro", "{:.3f}")
batang_dengan_seed(ax[1], "acc", "acc_sd", "accuracy", "Akurasi (sumbu y dipotong)", "Akurasi", "{:.3f}")
bawah = max(0.0, np.floor((d[["f1_macro", "accuracy"]].min().min() - 0.05) * 20) / 20)
for a in ax:
    a.set_ylim(bawah, 1.03)
ax[0].legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)
fig1.suptitle("Kualitas klasifikasi: rata-rata ± std antar-seed (titik = seed)", fontsize=14, fontweight="bold")
fig1.savefig(OUTPUT_DIR / f"grafik_1_akurasi_{KODE}.png")
plt.show()

# Gambar 2: biaya komputasi
fig2, ax = plt.subplots(1, 2, figsize=(15, 5.2), constrained_layout=True)
batang_dengan_seed(ax[0], "waktu", "waktu_sd", "train_time_s", "Waktu latih per run", "detik", "{:.0f}")
batang_dengan_seed(ax[1], "mem", "mem_sd", "peak_memory_mb", "Memori GPU puncak", "MB", "{:.0f}")
ax[0].legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)
fig2.suptitle("Biaya komputasi per strategi", fontsize=14, fontweight="bold")
fig2.savefig(OUTPUT_DIR / f"grafik_2_biaya_{KODE}.png")
plt.show()

# Gambar 3: trade-off F1 vs latensi (latensi dari benchmark satu sesi)
lat_stat = df_lat.groupby("backbone").latency_ms.agg(["mean", "std"]).fillna(0)
fig3, ax = plt.subplots(figsize=(10.5, 5.8), constrained_layout=True)
tanda = {"IndoBERT": "o", "mBERT": "s", "DistilmBERT": "^"}
SINGKAT = {"Full FT": "Full", "Partial freezing": "Partial", "Linear probing": "Probing"}
GESER = {"IndoBERT": (-11, 6, "right"), "mBERT": (11, -13, "left"), "DistilmBERT": (11, 6, "left")}
for _, r in agg.iterrows():
    x, sx = lat_stat.loc[KEY_BB[r.Backbone]]
    ax.errorbar(x, r.f1, xerr=sx, yerr=r.f1_sd, fmt=tanda[r.Backbone], ms=11, color=WARNA[r.Strategi],
                mec="black", mew=0.7, ecolor="#888888", capsize=2.5, zorder=3)
    dx, dy, ha = GESER[r.Backbone]
    ax.annotate(SINGKAT[r.Strategi], (x, r.f1), xytext=(dx, dy), textcoords="offset points", fontsize=9, ha=ha)
ax.set_xlabel("Latensi inferensi (ms/kalimat, batch = 1, satu sesi)")
ax.set_ylabel("F1 makro")
ax.set_title("Trade-off akurasi dan kecepatan (kiri atas lebih baik)", loc="left", pad=12)
from matplotlib.lines import Line2D
legenda = [Line2D([], [], marker=m, ls="", color="gray", mec="black", label=b) for b, m in tanda.items()] + \
          [Line2D([], [], marker="o", ls="", color=c, mec="black", label=s) for s, c in WARNA.items()]
ax.legend(handles=legenda, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.margins(x=0.1, y=0.1)
fig3.savefig(OUTPUT_DIR / f"grafik_3_tradeoff_{KODE}.png")
plt.show()

# Gambar 4: F1 per kelas
kelas = {"f1_negative": "negative", "f1_neutral": "neutral", "f1_positive": "positive"}
pivot = d.groupby(["Backbone", "Strategi"])[list(kelas)].mean().rename(columns=kelas)
pivot = pivot.reindex(pd.MultiIndex.from_product([URUT_BB, URUT_ST]))
pivot.index = [f"{b} · {s}" for b, s in pivot.index]
fig4, ax = plt.subplots(figsize=(7.5, 6), constrained_layout=True)
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1, linewidths=1, linecolor="white",
            cbar_kws={"label": "F1"}, annot_kws={"fontsize": 10}, ax=ax)
ax.set_title("F1 per kelas (rata-rata antar-seed)", loc="left", pad=12)
ax.set_xlabel("Kelas sentimen")
ax.set_ylabel("")
fig4.savefig(OUTPUT_DIR / f"grafik_4_f1_per_kelas_{KODE}.png")
plt.show()

## 12. Confusion Matrix

Confusion matrix menunjukkan **jenis kesalahan** yang tidak terlihat dari F1 tunggal: kelas mana yang tertukar dengan kelas mana.

- **Baris** adalah sentimen sebenarnya (*Actual*), **kolom** adalah prediksi model (*Predicted*).
- Setiap sel ditulis `jumlah (persentase)`. Persentase dihitung **per baris**, yaitu proporsi kalimat dari kelas aktual tersebut yang diprediksi ke kolom itu;
  nilai diagonal dengan demikian adalah *recall* tiap kelas. Warna mengikuti persentase ini dan memakai skala 0–100% yang sama di semua panel, sehingga antar-panel dapat dibandingkan.
- Jumlah adalah **rata-rata dari 3 seed** (dibulatkan), sehingga total tiap baris sama dengan banyaknya data validasi kelas itu.
- Gambar pertama menampilkan full fine-tuning untuk ketiga backbone; gambar kedua menampilkan seluruh 9 kombinasi. Data diambil dari prediksi yang tersimpan saat pelatihan.

Kelas *neutral* yang minoritas lazimnya paling sering salah diklasifikasikan; hal ini yang dicari pada gambar berikut.

In [ ]:
df_pred = pd.read_csv(berkas_prediksi)
NAMA_KELAS = ["negative", "neutral", "positive"]

def cm_rata_seed(bb, st):
    sub = df_pred[(df_pred.backbone == bb) & (df_pred.strategy == st)]
    mats = [confusion_matrix(g.y_true, g.y_pred, labels=[0, 1, 2]) for _, g in sub.groupby("seed")]
    return np.mean(mats, axis=0)

def gambar_cm(ax, bb, st, tampil_cbar=False):
    m = cm_rata_seed(bb, st)
    pct = m / m.sum(axis=1, keepdims=True) * 100
    label = np.array([[f"{m[i, j]:.0f} ({pct[i, j]:.1f}%)" for j in range(3)] for i in range(3)])
    sns.heatmap(pct, annot=label, fmt="", cmap="Blues", vmin=0, vmax=100, linewidths=1.5, linecolor="white",
                square=True, cbar=tampil_cbar, cbar_kws={"label": "% dari kelas aktual", "shrink": 0.8},
                xticklabels=NAMA_KELAS, yticklabels=NAMA_KELAS, annot_kws={"fontsize": 10}, ax=ax)
    ax.set_title(f"{NAMA_BB[bb]} · {NAMA_ST[st]}", pad=10)
    ax.set_xlabel("Predicted Sentiment")
    ax.set_ylabel("Actual Sentiment")

# Gambar 5: full fine-tuning, tiga backbone
fig5, axes = plt.subplots(1, 3, figsize=(17, 5.6), constrained_layout=True)
for k, bb in enumerate(CONFIG["backbones"]):
    gambar_cm(axes[k], bb, "full", tampil_cbar=(k == 2))
fig5.suptitle("Confusion matrix — Full Fine-Tuning (rata-rata 3 seed)", fontsize=14, fontweight="bold")
fig5.savefig(OUTPUT_DIR / f"grafik_5_confusion_full_{KODE}.png")
plt.show()

# Gambar 6: seluruh kombinasi
fig6, axes = plt.subplots(3, 3, figsize=(17, 16), constrained_layout=True)
for i, bb in enumerate(CONFIG["backbones"]):
    for j, st in enumerate(CONFIG["strategies"]):
        gambar_cm(axes[i, j], bb, st, tampil_cbar=(j == 2))
fig6.suptitle("Confusion matrix — seluruh kombinasi backbone × strategi (rata-rata 3 seed)",
              fontsize=14, fontweight="bold")
fig6.savefig(OUTPUT_DIR / f"grafik_6_confusion_semua_{KODE}.png")
plt.show()

## 13. Evaluasi Hipotesis H1–H3

Setiap hipotesis pada sel Rancangan diuji secara kuantitatif dari data run, agar kesimpulan bertumpu pada angka dan bukan kesan visual.

- **H1** (keunggulan IndoBERT): selisih rata-rata F1 makro full fine-tuning terhadap mBERT dan DistilmBERT, uji-t Welch, serta syarat bahwa selisih melampaui simpangan baku antar-seed.
- **H2** (efisiensi DistilmBERT terhadap mBERT): penghematan waktu latih (dari run full fine-tuning) dan latensi (dari benchmark satu sesi) minimal 35%,
  dengan penurunan F1 makro tidak lebih dari 3 poin.
- **H3** (pengaruh pembekuan): urutan F1 full ≥ partial > probing, partial mempertahankan ≥ 96% F1 full, dan menghemat waktu latih serta memori GPU ≥ 40%.

Status ditulis **TERDUKUNG / TIDAK TERDUKUNG** apa adanya; hipotesis yang gagal tetap dilaporkan dan dijelaskan. Dengan hanya tiga seed, uji-t
berdaya rendah dan hasilnya bersifat indikatif, sehingga p-value dibaca bersama besarnya selisih dan simpangan bakunya.

In [ ]:
from scipy import stats

def _g(bb, st, kolom):
    return df_hasil[(df_hasil.backbone == bb) & (df_hasil.strategy == st)][kolom].values

R = {}
# H1: IndoBERT vs mBERT dan DistilmBERT (full FT), F1 makro
for lawan in ("mbert", "distilmbert"):
    a, b = _g("indobert", "full", "f1_macro"), _g(lawan, "full", "f1_macro")
    selisih = a.mean() - b.mean()
    if len(a) < 2 or len(b) < 2:
        R[f"H1 indobert vs {lawan}"] = (f"selisih {selisih*100:+.2f} poin (seed < 2, tidak bisa diuji)", False)
        continue
    _, p = stats.ttest_ind(a, b, equal_var=False)
    sd = max(a.std(ddof=1), b.std(ddof=1))
    R[f"H1 indobert vs {lawan}"] = (
        f"selisih {selisih*100:+.2f} poin, std maks {sd*100:.2f}, p={p:.4f}",
        bool(selisih > 0 and p < 0.05 and selisih > sd))

# H2: DistilmBERT vs mBERT; waktu latih dari full FT, latensi dari benchmark satu sesi
m, dm = "mbert", "distilmbert"
hemat_latih = 1 - _g(dm, "full", "train_time_s").mean() / _g(m, "full", "train_time_s").mean()
lat_m = df_lat[df_lat.backbone == m].latency_ms.mean()
lat_d = df_lat[df_lat.backbone == dm].latency_ms.mean()
hemat_lat = 1 - lat_d / lat_m
drop_f1 = (_g(m, "full", "f1_macro").mean() - _g(dm, "full", "f1_macro").mean()) * 100
R["H2 hemat waktu latih >=35%"] = (f"{hemat_latih*100:.1f}%", hemat_latih >= 0.35)
R["H2 hemat latensi >=35% (satu sesi)"] = (f"{hemat_lat*100:.1f}% ({lat_m:.2f} -> {lat_d:.2f} ms)", hemat_lat >= 0.35)
R["H2 penurunan F1 <=3 poin"] = (f"{drop_f1:.2f} poin", drop_f1 <= 3.0)

# H3: hierarki dan efisiensi partial freezing (per backbone)
for bb in CONFIG["backbones"]:
    f, p_, l = (_g(bb, s, "f1_macro").mean() for s in ("full", "partial", "probing"))
    R[f"H3 {bb} hierarki full>=partial>probing"] = (f"{f:.4f} / {p_:.4f} / {l:.4f}", bool(f >= p_ > l))
    R[f"H3 {bb} partial>=96% F1 full"] = (f"{p_/f*100:.1f}%", p_ / f >= 0.96)
    hw = 1 - _g(bb, "partial", "train_time_s").mean() / _g(bb, "full", "train_time_s").mean()
    hm = 1 - _g(bb, "partial", "peak_memory_mb").mean() / _g(bb, "full", "peak_memory_mb").mean()
    R[f"H3 {bb} hemat waktu >=40%"] = (f"{hw*100:.1f}%", hw >= 0.40)
    R[f"H3 {bb} hemat memori >=40%"] = (f"{hm*100:.1f}%", hm >= 0.40)

tabel_hipotesis = pd.DataFrame([(k, v[0], "TERDUKUNG" if v[1] else "TIDAK TERDUKUNG") for k, v in R.items()],
                               columns=["pengujian", "nilai terukur", "status"])
simpan_csv_atomik(tabel_hipotesis, OUTPUT_DIR / f"evaluasi_hipotesis_{KODE}.csv")
display(tabel_hipotesis)

## 14. Jawaban Pertanyaan Riset Wajib

Tuliskan tiap pertanyaan sesuai bunyi resminya, lalu jawab dengan merujuk **angka dari tabel rekap (bagian 10), grafik (bagian 11–12), dan tabel hipotesis (bagian 13)**.
Jawaban yang baik menyebut selisih terhadap simpangan baku antar-seed, tidak menyimpulkan dari satu seed, dan mengakui hipotesis yang tidak terdukung.

**Pertanyaan 1:** _(tuliskan pertanyaan)_

**Jawaban:** _(tuliskan jawaban)_

**Pertanyaan 2:** _(tuliskan pertanyaan)_

**Jawaban:** _(tuliskan jawaban)_

**Pertanyaan 3:** _(tuliskan pertanyaan)_

**Jawaban:** _(tuliskan jawaban)_

## 15. Berkas Keluaran dan Pengumpulan

Berkas berikut dibuat otomatis di `MyDrive/Studio3_NLP/output/` (kode konfigurasi tercantum pada nama berkas):

| Berkas | Isi |
|:---|:---|
| `hasil_smsa_<kode>.csv` | satu baris per run (27 baris) dengan seluruh metrik |
| `rekap_agregat_smsa_<kode>.csv` | rata-rata ± std per kombinasi backbone × strategi |
| `prediksi_smsa_<kode>.csv` | prediksi tiap run, sumber confusion matrix |
| `latensi_sesi_<kode>.csv` | benchmark latensi satu sesi |
| `evaluasi_hipotesis_<kode>.csv` | status H1–H3 |
| `grafik_1` … `grafik_6` (PNG) | visualisasi performa, biaya, trade-off, F1 per kelas, dan confusion matrix |

**Yang dikumpulkan ke portal e-learning:** tautan notebook, PDF notebook yang sudah dijalankan penuh
(`STUDIO3_Kelompok[X]_[NamaAnggota1]_[NamaAnggota2].pdf`), berkas hasil CSV, serta slide (maksimal 15) dan laporan singkat 3–4 halaman pada hari presentasi.